# U2-08 · Redes generativas antagónicas y atención

*Programación de Redes Neuronales - Machine Learning (IFCD48) · Unidad 2, apartados 2.10 y 2.11*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con el botón ▶). Activa la GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*) para la GAN de dígitos: sin ella tarda unos 20 minutos.

> Datos: MNIST, de Y. LeCun, C. Cortes y C. J. C. Burges (CC BY-SA 3.0). Keras lo descarga en la primera ejecución.

## 1. Una GAN en una dimensión

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

def reales(n, rng):                  # dos picos: en -2 y en +2
    centro = rng.choice([-2.0, 2.0], n)
    return (centro + rng.normal(0, 0.5, n)).astype("float32")[:, None]

def red(salida):
    return keras.Sequential([keras.Input(shape=(1,)),
                             layers.Dense(32, "relu"),
                             layers.Dense(32, "relu"),
                             layers.Dense(1, activation=salida)])

keras.utils.set_random_seed(0)
rng = np.random.default_rng(0)
G, D = red(None), red(None)          # generador y discriminador
opt_g = keras.optimizers.Adam(1e-3)
opt_d = keras.optimizers.Adam(1e-3)
bce = keras.losses.BinaryCrossentropy(from_logits=True)

@tf.function
def paso(x_real):
    z = tf.random.normal((len(x_real), 1))
    with tf.GradientTape() as cd, tf.GradientTape() as cg:
        x_falso = G(z)
        d_real, d_falso = D(x_real), D(x_falso)
        perd_d = (bce(tf.ones_like(d_real), d_real)       # 1 = real
                  + bce(tf.zeros_like(d_falso), d_falso)) # 0 = falso
        perd_g = bce(tf.ones_like(d_falso), d_falso)      # engañar
    opt_d.apply_gradients(zip(cd.gradient(perd_d, D.weights),
                              D.weights))
    opt_g.apply_gradients(zip(cg.gradient(perd_g, G.weights),
                              G.weights))
    return perd_d, perd_g

for t in range(1, 3001):
    pd_, pg = paso(reales(256, rng))
    if t in (1, 500, 1000, 3000):
        m = G(tf.random.normal((5000, 1))).numpy()[:, 0]
        print(f"paso {t:4d}: pérdida D {float(pd_):.3f}, "
              f"G {float(pg):.3f} | izquierda {np.mean(m < 0):.2f}, "
              f"desv. {m.std():.2f}")

x = reales(5000, rng)[:, 0]
plt.hist(x, 60, density=True, alpha=0.5, label="datos reales")
plt.hist(m, 60, density=True, alpha=0.5, label="generados")
plt.legend(); plt.show()  # plt.savefig("gan1d.pdf")
for nombre, v in [("reales", x), ("generados", m)]:
    print(f"{nombre}: desv. dentro de cada pico "
          f"{v[v < 0].std():.2f} y {v[v > 0].std():.2f}")
print(f"equilibrio teórico de D: 2 ln 2 = {2 * np.log(2):.3f}")

**Prueba.** Cambia los datos reales a tres picos (en −3, 0 y 3). ¿Los cubre todos el generador?

## 2. Una GAN que genera dígitos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

(x, y), _ = keras.datasets.mnist.load_data()
x = (x.reshape(-1, 784).astype("float32") - 127.5) / 127.5  # [-1, 1]

keras.utils.set_random_seed(0)
G = keras.Sequential([keras.Input(shape=(64,)),
                      layers.Dense(256), layers.LeakyReLU(0.2),
                      layers.Dense(512), layers.LeakyReLU(0.2),
                      layers.Dense(784, activation="tanh")])
D = keras.Sequential([keras.Input(shape=(784,)),
                      layers.Dense(512), layers.LeakyReLU(0.2),
                      layers.Dense(256), layers.LeakyReLU(0.2),
                      layers.Dense(1)])
opt_g = keras.optimizers.Adam(2e-4, beta_1=0.5)
opt_d = keras.optimizers.Adam(2e-4, beta_1=0.5)
bce = keras.losses.BinaryCrossentropy(from_logits=True)

@tf.function
def paso(x_real):
    z = tf.random.normal((tf.shape(x_real)[0], 64))
    with tf.GradientTape() as cd, tf.GradientTape() as cg:
        d_real, d_falso = D(x_real), D(G(z))
        perd_d = (bce(tf.ones_like(d_real), d_real)
                  + bce(tf.zeros_like(d_falso), d_falso))
        perd_g = bce(tf.ones_like(d_falso), d_falso)
    opt_d.apply_gradients(zip(cd.gradient(perd_d, D.weights),
                              D.weights))
    opt_g.apply_gradients(zip(cg.gradient(perd_g, G.weights),
                              G.weights))
    return perd_d, perd_g

z_fijo = tf.random.normal((8, 64))       # los mismos 8 puntos siempre
lotes = tf.data.Dataset.from_tensor_slices(x)
lotes = lotes.shuffle(60_000).batch(128)
filas = []
for epoca in range(1, 21):
    for lote in lotes:
        perd_d, perd_g = paso(lote)
    if epoca in (1, 5, 20):
        filas.append(G(z_fijo).numpy().reshape(8, 28, 28))
        print(f"época {epoca:2d}: pérdida D {float(perd_d):.3f}, "
              f"G {float(perd_g):.3f}")

# ¿qué dígitos genera? Los clasifica una red entrenada aparte
juez = keras.Sequential([keras.Input(shape=(784,)),
                         layers.Dense(128, activation="relu"),
                         layers.Dense(10, activation="softmax")])
juez.compile("adam", "sparse_categorical_crossentropy")
juez.fit(x, y, epochs=3, verbose=0)
p = juez.predict(G(tf.random.normal((5000, 64))), verbose=0)
print("dígitos generados:", np.bincount(p.argmax(1), minlength=10))
print(f"con confianza > 0.9: {np.mean(p.max(1) > 0.9):.2f}")

fig, ejes = plt.subplots(3, 8, figsize=(4.4, 1.8))
for f, (fila, ep) in enumerate(zip(filas, [1, 5, 20])):
    for c in range(8):
        ejes[f, c].imshow(fila[c]); ejes[f, c].axis("off")
    ejes[f, 0].set_title(f"época {ep}", loc="left", fontsize=6)
plt.show()  # fig.savefig("gan_mnist.pdf")

**Prueba.** Entrena 40 épocas. ¿Se reparten mejor los dígitos generados? ¿Sube la proporción con confianza > 0.9?

## 3. La atención, a mano

In [ ]:
import numpy as np

def atencion(Q, K, V):
    """Atención de producto escalar: cada consulta reparte su peso
    entre las claves parecidas y devuelve la media de sus valores."""
    s = Q @ K.T / np.sqrt(K.shape[1])          # parecido
    w = np.exp(s - s.max(1, keepdims=True))
    w /= w.sum(1, keepdims=True)                # softmax por fila
    return w @ V, w

K = np.array([[1., 0, 0, 0], [0, 1, 0, 0],     # 4 posiciones
              [0, 0, 1, 0], [0, 0, 0, 1]]) * 3  # de memoria
V = np.array([[10.], [20.], [30.], [40.]])      # lo que guardan
Q = np.array([[0, 0, 3., 0],                    # busca la 3.ª
              [0, 3., 3., 0]])                  # busca la 2.ª y 3.ª
salida, w = atencion(Q, K, V)
print("pesos:\n", w.round(3))
print("salida:", salida.ravel().round(2))

## 4. Atención frente a LSTM en el problema de la suma

La primera celda guarda como archivo el código del apartado 2.8, que la segunda importa. Tarda unos minutos.

In [ ]:
%%writefile v_memoria.py
import numpy as np
import keras
from keras import layers

def sumas(n, largo, semilla):
    """Problema de la suma: hay que sumar los dos valores marcados,
    uno en la primera mitad de la secuencia y otro en la segunda."""
    rng = np.random.default_rng(semilla)
    v = rng.uniform(0, 1, (n, largo))
    m = np.zeros((n, largo))
    m[np.arange(n), rng.integers(0, largo // 2, n)] = 1
    m[np.arange(n), rng.integers(largo // 2, largo, n)] = 1
    x = np.stack([v, m], axis=-1).astype("float32")
    return x, (v * m).sum(1).astype("float32")

if __name__ == "__main__":          # solo al ejecutar este archivo
    for largo in [10, 50, 100]:
        x, y = sumas(10_000, largo, 0)
        xp, yp = sumas(1000, largo, 1)
        for tipo in [layers.SimpleRNN, layers.LSTM]:
            keras.utils.set_random_seed(0)
            red = keras.Sequential([keras.Input(shape=(largo, 2)),
                                    tipo(32), layers.Dense(1)])
            red.compile("adam", "mse")
            red.fit(x, y, epochs=20, batch_size=64, verbose=0)
            ecm = red.evaluate(xp, yp, verbose=0)
            print(f"{largo:3d} pasos, {tipo.__name__:<9}: error "
                  f"{ecm:.4f}")
    print(f"sin memoria (responder siempre 1): "
          f"{np.mean((yp - 1) ** 2):.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers, ops
from v_memoria import sumas                # el problema de 2.8

largo = 150
x, y = sumas(10_000, largo, 0)
xp, yp = sumas(1000, largo, 1)

def con_atencion():
    entrada = keras.Input(shape=(largo, 2))
    h = layers.Dense(32, activation="tanh")(entrada)   # cada paso
    puntos = layers.Dense(1)(h)                        # relevancia
    pesos = layers.Softmax(axis=1, name="pesos")(puntos)  # suman 1
    contexto = ops.sum(pesos * h, axis=1)              # media pond.
    oculta = layers.Dense(32, activation="relu")(contexto)
    return keras.Model(entrada, layers.Dense(1)(oculta))

def con_lstm():
    return keras.Sequential([keras.Input(shape=(largo, 2)),
                             layers.LSTM(32), layers.Dense(1)])

for nombre, construir in [("atención", con_atencion),
                          ("LSTM", con_lstm)]:
    keras.utils.set_random_seed(0)
    m = construir()
    m.compile("adam", "mse")
    m.fit(x, y, epochs=20, batch_size=64, verbose=0)
    print(f"{largo} pasos, {nombre:<8}: error "
          f"{m.evaluate(xp, yp, verbose=0):.2e}")
    if nombre == "atención":
        ver = keras.Model(m.inputs, m.get_layer("pesos").output)
        w = ver.predict(xp[:1], verbose=0)[0, :, 0]

marcas = np.flatnonzero(xp[0, :, 1])
print("posiciones marcadas:", marcas,
      f"| peso total sobre ellas: {w[marcas].sum():.3f}")
plt.plot(w); plt.plot(marcas, w[marcas], "o")
plt.xlabel("paso"); plt.ylabel("peso de atención")
plt.show()  # plt.savefig("atencion.pdf")

**Ejercicio.** Prueba la red con atención con secuencias de 500 pasos. ¿Sigue encontrando las dos posiciones marcadas?

> Goodfellow, I. et al. (2020). Generative adversarial networks. *Communications of the ACM*, 63(11), 139–144. https://doi.org/10.1145/3422622
>
> Vaswani, A. et al. (2017). Attention is all you need. arXiv:1706.03762. https://doi.org/10.48550/arXiv.1706.03762